# Thai Math VQA Full Pipeline

Self-contained notebook for the Thai Math VQA challenge.

This notebook includes the full implementation in notebook cells. It does not import any local
`src.math_vqa` pipeline module. The Gemma, Qwen, and prior predictor classes are defined inline in
this notebook. External package imports such as `pandas`, `PIL`, `transformers`, and `qwen_vl_utils`
are still required to run those cells.

Workflow:

1. Load `train.csv`, `test.csv`, and `sample_submission.csv` from the local zip archive.
2. Inspect dataset shape and answer formats.
3. Preprocess images while preserving Thai text, symbols, and diagrams.
4. Run validation with either `prior`, `gemma`, or `qwen`.
5. Generate `submission_math_vqa.csv`.

Default real VLM backend: Gemma 3 (`google/gemma-3-4b-it`).



In [ ]:
from __future__ import annotations

import csv
import json
import re
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

pd.set_option("display.max_colwidth", 120)


REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().parents[1]


DEFAULT_ARCHIVE = Path("dataset/super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen.zip")
DEFAULT_EXTRACTED = Path("dataset/Individual-Test-Math-VQA-Challenge")
DEFAULT_GEMMA_MODEL = "google/gemma-3-4b-it"
DEFAULT_QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
DEFAULT_OUTPUT_DIR = Path("outputs/math_vqa")


PROMPT_VARIANTS = {
    "direct": (
        "You are solving a Thai math problem from an image. Read every visible Thai word, "
        "number, label, unit, and math symbol. Solve the problem carefully. Return only the "
        "final answer exactly as it should appear in a CSV. If the answer needs a Thai unit, "
        "include it. If the exact answer is a fraction, radical, pi expression, or equation, "
        "preserve exact math notation. Do not include explanation."
    ),
    "ocr_mindful": (
        "Solve this Thai mathematics image. Pay special attention to small labels, units, "
        "fractions, radicals, exponents, angle marks, and diagram annotations. The final output "
        "must be the answer only, with no reasoning text."
    ),
    "format_strict": (
        "Find the final answer for the math question in the image. Output one short string only. "
        "Do not add markdown, punctuation, explanation, or a label such as 'answer:'. Keep Thai "
        "units and exact symbolic forms when needed."
    ),
}


@dataclass(frozen=True)
class DatasetPaths:
    archive: Path
    extracted_root: Path


def normalize_answer(value: Any) -> str:
    text = "" if value is None else str(value)
    text = text.replace("\ufeff", "").strip()
    text = re.sub(r"^```(?:text|csv|json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip().strip("\"'`")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ",", text)
    text = re.sub(r"\s*=\s*", "=", text)
    text = re.sub(r"^(answer|final answer|คำตอบ)\s*[:：]\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


def answer_type(answer: str) -> str:
    answer = normalize_answer(answer)
    if answer == "<n/a>":
        return "na_literal"
    if re.fullmatch(r"[-+]?\d+", answer):
        return "integer"
    if re.fullmatch(r"[-+]?\d+\.\d+", answer):
        return "decimal"
    if any(token in answer for token in ("\\frac", "\\sqrt", "$", "\\pi")):
        return "latex"
    if re.search(r"[ก-๙]", answer):
        return "thai_text_or_unit"
    return "expression_or_text"


class MathVQADataset:
    def __init__(self, paths: DatasetPaths) -> None:
        self.paths = paths
        self.archive = paths.archive
        self.extracted_root = paths.extracted_root

    def read_csv(self, split: str) -> pd.DataFrame:
        filename = {
            "train": "train.csv",
            "test": "test.csv",
            "sample": "sample_submission.csv",
        }[split]

        extracted_file = self.extracted_root / filename
        if extracted_file.exists():
            dtype = {"answer": str} if split != "test" else None
            return pd.read_csv(extracted_file, dtype=dtype)

        if not self.archive.exists():
            raise FileNotFoundError(
                f"Could not find {filename}. Checked {extracted_file} and {self.archive}."
            )
        with zipfile.ZipFile(self.archive) as zf:
            with zf.open(filename) as f:
                dtype = {"answer": str} if split != "test" else None
                return pd.read_csv(f, dtype=dtype)

    def image_source(self, image_path: str) -> tuple[str, Path | None]:
        direct = self.extracted_root / image_path
        nested = self.extracted_root / "images" / image_path
        if direct.exists():
            return "file", direct
        if nested.exists():
            return "file", nested
        if self.archive.exists():
            return "zip", None
        raise FileNotFoundError(f"Image not found for CSV path {image_path!r}")

    def open_image(self, image_path: str) -> Image.Image:
        source, path = self.image_source(image_path)
        if source == "file":
            assert path is not None
            return Image.open(path).convert("RGB")

        zip_member = f"images/{image_path}".replace("\\", "/")
        with zipfile.ZipFile(self.archive) as zf:
            with zf.open(zip_member) as f:
                return Image.open(f).convert("RGB")


def preprocess_image(
    image: Image.Image,
    *,
    max_side: int = 1800,
    pad_multiple: int = 28,
    enhance: bool = True,
) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    scale = min(1.0, max_side / max(width, height))
    if scale < 1.0:
        new_size = (round(width * scale), round(height * scale))
        image = image.resize(new_size, Image.Resampling.LANCZOS)

    if enhance:
        image = ImageEnhance.Contrast(image).enhance(1.08)
        image = ImageEnhance.Sharpness(image).enhance(1.12)
        image = image.filter(ImageFilter.UnsharpMask(radius=1.0, percent=80, threshold=3))

    width, height = image.size
    padded_width = ((width + pad_multiple - 1) // pad_multiple) * pad_multiple
    padded_height = ((height + pad_multiple - 1) // pad_multiple) * pad_multiple
    if (padded_width, padded_height) == (width, height):
        return image

    canvas = Image.new("RGB", (padded_width, padded_height), "white")
    canvas.paste(image, ((padded_width - width) // 2, (padded_height - height) // 2))
    return canvas


def build_preprocessed_image(
    dataset: MathVQADataset,
    image_path: str,
    cache_dir: Path,
    *,
    max_side: int,
    enhance: bool,
) -> Path:
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_path = cache_dir / Path(image_path).name
    if out_path.exists():
        return out_path
    image = dataset.open_image(image_path)
    processed = preprocess_image(image, max_side=max_side, enhance=enhance)
    processed.save(out_path, quality=96, optimize=True)
    return out_path


class JsonlCache:
    def __init__(self, path: Path) -> None:
        self.path = path
        self.rows: dict[str, dict[str, Any]] = {}
        if path.exists():
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        row = json.loads(line)
                        self.rows[str(row["id"])] = row

    def get(self, sample_id: int | str) -> dict[str, Any] | None:
        return self.rows.get(str(sample_id))

    def append(self, row: dict[str, Any]) -> None:
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        self.rows[str(row["id"])] = row


class PriorPredictor:
    def __init__(self, train_df: pd.DataFrame) -> None:
        counts = Counter(normalize_answer(v) for v in train_df["answer"].tolist())
        self.answer = counts.most_common(1)[0][0] if counts else "1"

    def predict(self, image_path: Path, prompt: str) -> str:
        _ = image_path, prompt
        return self.answer


class GemmaVLPredictor:
    def __init__(
        self,
        model_id: str,
        *,
        device_map: str = "auto",
        torch_dtype: str = "auto",
        max_new_tokens: int = 96,
    ) -> None:
        try:
            import torch
            from transformers import AutoProcessor
        except ImportError as exc:
            raise RuntimeError(
                "Gemma inference requires the math_vqa dependency group. "
                "Install/sync it with uv before running model inference."
            ) from exc

        try:
            from transformers import Gemma3ForConditionalGeneration as ModelClass
        except ImportError:
            try:
                from transformers import AutoModelForImageTextToText as ModelClass
            except ImportError:
                from transformers import AutoModelForVision2Seq as ModelClass

        self.torch = torch
        self.max_new_tokens = max_new_tokens
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self.model = ModelClass.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
        )
        self.model.eval()

    def predict(self, image_path: Path, prompt: str) -> str:
        image = Image.open(image_path).convert("RGB")
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = inputs.to(self.model.device)
        input_len = inputs["input_ids"].shape[-1]
        with self.torch.inference_mode():
            generated_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        output_text = self.processor.decode(
            generated_ids[0][input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        return normalize_answer(output_text)


class QwenVLPredictor:
    def __init__(
        self,
        model_id: str,
        *,
        device_map: str = "auto",
        torch_dtype: str = "auto",
        max_new_tokens: int = 96,
    ) -> None:
        try:
            import torch
            from qwen_vl_utils import process_vision_info
            from transformers import AutoProcessor
        except ImportError as exc:
            raise RuntimeError(
                "Qwen inference requires the math_vqa dependency group. "
                "Install/sync it with uv before running model inference."
            ) from exc

        try:
            from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
        except ImportError:
            try:
                from transformers import AutoModelForImageTextToText as ModelClass
            except ImportError:
                from transformers import AutoModelForVision2Seq as ModelClass

        self.torch = torch
        self.process_vision_info = process_vision_info
        self.max_new_tokens = max_new_tokens
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self.model = ModelClass.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
        )
        self.model.eval()

    def predict(self, image_path: Path, prompt: str) -> str:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": str(image_path)},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = self.process_vision_info(messages)
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(self.model.device)
        with self.torch.inference_mode():
            generated_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids, strict=True)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        return normalize_answer(output_text)


def choose_answer(candidates: list[dict[str, str]]) -> str:
    normalized = []
    for candidate in candidates:
        answer = normalize_answer(candidate["answer"])
        if answer:
            normalized.append(answer)
    if not normalized:
        return ""
    counts = Counter(normalized)
    return counts.most_common(1)[0][0]


def run_predictions(
    dataset: MathVQADataset,
    rows: pd.DataFrame,
    predictor: Any,
    *,
    cache_path: Path,
    image_cache_dir: Path,
    prompt_variants: list[str],
    max_side: int,
    enhance: bool,
    limit: int | None = None,
) -> pd.DataFrame:
    cache = JsonlCache(cache_path)
    outputs: list[dict[str, Any]] = []
    selected = rows.head(limit) if limit else rows

    for idx, row in selected.reset_index(drop=True).iterrows():
        sample_id = int(row["id"])
        cached = cache.get(sample_id)
        if cached:
            outputs.append(cached)
            continue

        image_path = build_preprocessed_image(
            dataset,
            str(row["image_path"]),
            image_cache_dir,
            max_side=max_side,
            enhance=enhance,
        )
        candidates = []
        for variant in prompt_variants:
            prompt = PROMPT_VARIANTS[variant]
            answer = predictor.predict(image_path, prompt)
            candidates.append({"variant": variant, "answer": normalize_answer(answer)})

        final_answer = choose_answer(candidates)
        out_row = {
            "id": sample_id,
            "image_path": row["image_path"],
            "answer": final_answer,
            "candidates": candidates,
        }
        if "answer" in row:
            out_row["target"] = normalize_answer(row["answer"])
        cache.append(out_row)
        outputs.append(out_row)
        print(f"[{idx + 1}/{len(selected)}] id={sample_id} answer={final_answer}", flush=True)

    return pd.DataFrame(outputs)


def evaluate_predictions(pred_df: pd.DataFrame) -> dict[str, Any]:
    if "target" not in pred_df.columns:
        raise ValueError("Prediction dataframe has no target column.")
    pred = pred_df["answer"].map(normalize_answer)
    target = pred_df["target"].map(normalize_answer)
    correct = pred == target
    by_type: dict[str, dict[str, Any]] = {}
    typed_predictions = pred_df.assign(_correct=correct, _type=target.map(answer_type))
    for label, group in typed_predictions.groupby("_type"):
        by_type[label] = {
            "count": int(len(group)),
            "exact_match": float(group["_correct"].mean()),
        }
    return {
        "count": int(len(pred_df)),
        "exact_match": float(correct.mean()) if len(correct) else 0.0,
        "correct": int(correct.sum()),
        "by_type": by_type,
    }


def write_submission(pred_df: pd.DataFrame, sample_df: pd.DataFrame, output_path: Path) -> None:
    answer_map = {int(row.id): normalize_answer(row.answer) for row in pred_df.itertuples()}
    missing = [
        int(sample_id)
        for sample_id in sample_df["id"].tolist()
        if int(sample_id) not in answer_map
    ]
    if missing:
        raise ValueError(f"Missing predictions for {len(missing)} sample IDs, e.g. {missing[:10]}")
    out = sample_df[["id"]].copy()
    out["answer"] = out["id"].map(lambda x: answer_map[int(x)])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_path, index=False, quoting=csv.QUOTE_MINIMAL)


def split_train(df: pd.DataFrame, valid_size: int, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    if valid_size <= 0 or valid_size >= len(df):
        return df, df
    sampled = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    valid = sampled.head(valid_size)
    train = sampled.iloc[valid_size:].reset_index(drop=True)
    return train, valid


def safe_tag(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "-", value)
    value = value.strip("-._")
    return value or "run"


In [ ]:
# Core paths
ARCHIVE = REPO_ROOT / DEFAULT_ARCHIVE
EXTRACTED_ROOT = REPO_ROOT / DEFAULT_EXTRACTED
OUTPUT_DIR = REPO_ROOT / DEFAULT_OUTPUT_DIR
SUBMISSION_PATH = REPO_ROOT / "submission_math_vqa.csv"

# Inference settings
# Use "prior" for a fast end-to-end smoke test.
# Use "gemma" for Gemma 3 vision inference.
# Use "qwen" for Qwen2.5-VL inference.
BACKEND = "gemma"
MODEL_ID_BY_BACKEND = {
    "gemma": DEFAULT_GEMMA_MODEL,
    "qwen": DEFAULT_QWEN_MODEL,
    "prior": "answer-prior",
}
MODEL_ID = MODEL_ID_BY_BACKEND[BACKEND]
DEVICE_MAP = "auto"
TORCH_DTYPE = "auto"
MAX_NEW_TOKENS = 96

# Pipeline settings
PROMPT_VARIANT_NAMES = ["direct", "ocr_mindful", "format_strict"]
MAX_SIDE = 1800
ENHANCE_IMAGE = True
VALID_SIZE = 56
SEED = 42

# Safety controls
# Keep small for smoke tests. Set to None for full validation/submission.
LIMIT = 3
RUN_VALIDATION = True
RUN_SUBMISSION = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TAG = safe_tag(
    "__".join(
        [
            BACKEND,
            MODEL_ID,
            "prompts-" + "-".join(PROMPT_VARIANT_NAMES),
            f"maxside-{MAX_SIDE}",
            "enhance-1" if ENHANCE_IMAGE else "enhance-0",
        ]
    )
)

print(f"Backend: {BACKEND}")
print(f"Model: {MODEL_ID}")
print(f"Run tag: {RUN_TAG}")
print(f"Output directory: {OUTPUT_DIR}")



In [ ]:
dataset = MathVQADataset(DatasetPaths(archive=ARCHIVE, extracted_root=EXTRACTED_ROOT))

train_df = dataset.read_csv("train")
test_df = dataset.read_csv("test")
sample_df = dataset.read_csv("sample")

print("train:", train_df.shape)
print("test:", test_df.shape)
print("sample_submission:", sample_df.shape)

display(train_df.head())
display(test_df.head())
display(sample_df.head())



In [ ]:
profile = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "sample_rows": len(sample_df),
    "unique_train_ids": train_df["id"].nunique(),
    "unique_test_ids": test_df["id"].nunique(),
    "id_overlap": len(set(train_df["id"]) & set(test_df["id"])),
    "answer_unique": train_df["answer"].map(normalize_answer).nunique(),
}

answer_profile = (
    train_df.assign(
        answer_norm=train_df["answer"].map(normalize_answer),
        answer_type=train_df["answer"].map(answer_type),
    )
    .groupby("answer_type")
    .agg(count=("id", "count"), unique_answers=("answer_norm", "nunique"))
    .sort_values("count", ascending=False)
)

top_answers = (
    train_df["answer"]
    .map(normalize_answer)
    .value_counts()
    .head(20)
    .rename_axis("answer")
    .reset_index(name="count")
)

display(pd.DataFrame([profile]).T.rename(columns={0: "value"}))
display(answer_profile)
display(top_answers)



In [ ]:
preview_rows = pd.concat([train_df.head(2), test_df.head(2)], ignore_index=True)
preview_dir = OUTPUT_DIR / "images_preprocessed"

preview_paths = []
for row in preview_rows.itertuples():
    processed_path = build_preprocessed_image(
        dataset,
        row.image_path,
        preview_dir,
        max_side=MAX_SIDE,
        enhance=ENHANCE_IMAGE,
    )
    image = dataset.open_image(row.image_path)
    preview_paths.append(
        {
            "id": row.id,
            "csv_image_path": row.image_path,
            "original_size": image.size,
            "processed_path": str(processed_path.relative_to(REPO_ROOT)),
        }
    )

display(pd.DataFrame(preview_paths))



In [ ]:
if BACKEND == "prior":
    predictor = PriorPredictor(train_df)
elif BACKEND == "gemma":
    predictor = GemmaVLPredictor(
        MODEL_ID,
        device_map=DEVICE_MAP,
        torch_dtype=TORCH_DTYPE,
        max_new_tokens=MAX_NEW_TOKENS,
    )
elif BACKEND == "qwen":
    predictor = QwenVLPredictor(
        MODEL_ID,
        device_map=DEVICE_MAP,
        torch_dtype=TORCH_DTYPE,
        max_new_tokens=MAX_NEW_TOKENS,
    )
else:
    raise ValueError(f"Unsupported backend: {BACKEND}")

print(type(predictor).__name__)



In [ ]:
if RUN_VALIDATION:
    fit_df, valid_df = split_train(train_df, VALID_SIZE, SEED)
    validation_predictor = PriorPredictor(fit_df) if BACKEND == "prior" else predictor
    valid_pred = run_predictions(
        dataset,
        valid_df,
        validation_predictor,
        cache_path=OUTPUT_DIR / f"valid_predictions_{RUN_TAG}.jsonl",
        image_cache_dir=OUTPUT_DIR / "images_preprocessed",
        prompt_variants=PROMPT_VARIANT_NAMES,
        max_side=MAX_SIDE,
        enhance=ENHANCE_IMAGE,
        limit=LIMIT,
    )
    valid_metrics = evaluate_predictions(valid_pred)
    metrics_path = OUTPUT_DIR / f"valid_metrics_{RUN_TAG}.json"
    metrics_path.write_text(
        json.dumps(valid_metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(valid_metrics, ensure_ascii=False, indent=2))
    display(valid_pred[["id", "image_path", "target", "answer", "candidates"]].head(20))
else:
    print("Validation skipped.")



In [ ]:
if RUN_SUBMISSION:
    test_pred = run_predictions(
        dataset,
        test_df,
        predictor,
        cache_path=OUTPUT_DIR / f"test_predictions_{RUN_TAG}.jsonl",
        image_cache_dir=OUTPUT_DIR / "images_preprocessed",
        prompt_variants=PROMPT_VARIANT_NAMES,
        max_side=MAX_SIDE,
        enhance=ENHANCE_IMAGE,
        limit=LIMIT,
    )

    if LIMIT is None:
        write_submission(test_pred, sample_df, SUBMISSION_PATH)
        print(f"Wrote full submission: {SUBMISSION_PATH}")
        display(pd.read_csv(SUBMISSION_PATH).head())
    else:
        preview_path = OUTPUT_DIR / "submission_preview.csv"
        test_pred[["id", "answer"]].to_csv(preview_path, index=False)
        print(f"LIMIT is set, so wrote preview predictions only: {preview_path}")
        display(test_pred[["id", "answer", "candidates"]].head(20))
else:
    print("Submission skipped. Set RUN_SUBMISSION=True to generate predictions for test.csv.")



## Full Gemma Submission Run

For a real Gemma submission, edit the config cell:

```python
BACKEND = "gemma"
MODEL_ID = DEFAULT_GEMMA_MODEL  # "google/gemma-3-4b-it"
LIMIT = None
RUN_VALIDATION = True
RUN_SUBMISSION = True
```

Then run all cells. The notebook writes:

- validation cache: `outputs/math_vqa/valid_predictions_<run-tag>.jsonl`
- validation metrics: `outputs/math_vqa/valid_metrics_<run-tag>.json`
- test cache: `outputs/math_vqa/test_predictions_<run-tag>.jsonl`
- final submission: `submission_math_vqa.csv`

Gemma model access may require accepting the model license on Hugging Face and authenticating with `HF_TOKEN`.

To use a fast smoke test without loading a VLM:

```python
BACKEND = "prior"
LIMIT = 3
RUN_VALIDATION = True
RUN_SUBMISSION = False
```

To switch to Qwen:

```python
BACKEND = "qwen"
MODEL_ID = DEFAULT_QWEN_MODEL
```

